# **Overview**


Home Advantage in Football — Bias or Real?

This project investigates whether the well-known “home advantage” in football is a genuine effect of playing at home before a supportive crowd, or whether it is largely explained by differences in team strength.
A Decision Tree Classifier is a supervised machine learning model that predicts a category by asking a series of yes/no questions about the data. It is structured like a flowchart: starting from a single top node, the data is repeatedly split into branches until a final prediction is reached.
is trained to predict the outcome of a match, and the model’s feature importances are analysed to determine which factors most influence the result.

The dataset consists of 50,000 football matches drawn from several major leagues (English Premier League, La Liga, Serie A, and Bundesliga, among others). Each record describes the conditions of a single match, including:
Team strength: Elo ratings for the home and away teams, and their difference.
Crowd conditions: stadium capacity, attendance percentage, crowd size, and whether fans were present (including empty “ghost games”).
Travel factors: the away team’s travel distance, time zones crossed, and whether the fixture is a local derby.
Match context: venue familiarity and referee susceptibility to crowd pressure.
The target variable is the match result, classified as home win, draw, or away win.


# **(1) Import libraries, load dataset**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
import os
import kagglehub

# Download the dataset (cached after the first run)
path = kagglehub.dataset_download("sergionefedov/home-advantage-in-football-bias-or-real")

# Show every file in the dataset
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), path))

# Load the first CSV found
csv_files = [os.path.join(root, f)
             for root, dirs, files in os.walk(path)
             for f in files if f.endswith(".csv")]

df = pd.read_csv(csv_files[0])
print("\nLoaded:", os.path.basename(csv_files[0]))
df.head()

Using Colab cache for faster access to the 'home-advantage-in-football-bias-or-real' dataset.
data_dictionary_home_advantage.csv
home_advantage_matches.csv

Loaded: data_dictionary_home_advantage.csv


,column,type,description
0,match_id,string,Unique match id
1,league,string,"Competition (EPL, La_Liga, Serie_A, Bundesliga..."
2,elo_home,int,EARNED: home team strength (Elo-like)
3,elo_away,int,EARNED: away team strength (Elo-like)
4,elo_diff,int,elo_home - elo_away (positive favors home on m...


# **(2) Apply EDA**.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   column       25 non-null     object
 1   type         25 non-null     object
 2   description  25 non-null     object
dtypes: object(3)
memory usage: 732.0+ bytes


In [ ]:
df.describe().T.round(2)

,count,unique,top,freq
column,25,25,match_id,1
type,25,3,int,18
description,25,25,Unique match id,1


**Observations**
- 50,000 rows (one per match) and 25 columns.
- 23 columns are numeric. Only `match_id`, `league` and `home_result` are text.
- `elo_diff` is centred around 0 (mean ≈ 1), so home and away teams are evenly matched in strength on average.
- Several columns are **binary flags** (0/1): `crowd_present`, `is_derby`, `red_home`, `red_away`.

# 2.2 Identify features and target variable
Target variable: home_result, the match outcome from the home team's view (home_win, draw, away_win).

Columns fall into four groups. This matters because some columns are only known after the match. Using them would let the model "cheat" (data leakage).

In [ ]:
target = "home_result"

pre_match_features = ["league", "elo_home", "elo_away", "elo_diff", "crowd_present",
                      "stadium_capacity", "attendance_pct", "crowd_size",
                      "travel_distance_km", "timezones_crossed", "is_derby",
                      "venue_familiarity", "referee_bias_susceptibility"]
in_match_stats = ["fouls_home", "fouls_away", "yellow_home", "yellow_away", "red_home", "red_away"]
leakage_cols   = ["goals_home", "goals_away", "home_goal_diff", "home_points"]

print("Target:", target)
print("Number of pre-match features:", len(pre_match_features))

Target: home_result
Number of pre-match features: 13


In [ ]:
# Missing values and duplicates
missing = df.isna().sum()
print("Total missing values:", missing.sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate match_id:", df["match_id"].duplicated().sum())

Total missing values: 0
Duplicate rows: 0


KeyError: 'match_id'


**Observations:** there are no missing values and no duplicates, so no imputation or row removal is needed in preprocessing.

In [ ]:
# Target distribution (count plot)
order = ["home_win", "draw", "away_win"]
ax = sns.countplot(data=df, x=target, order=order, palette="Set2")
for p in ax.patches:
    ax.annotate(f"{p.get_height()/len(df):.1%}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Match result distribution (home team's perspective)")
ax.set_xlabel(""); ax.set_ylabel("Number of matches")
plt.show()

df[target].value_counts(normalize=True).round(3)

**Observations**

Home teams win ≈47% of matches, versus ≈29% away wins. This is the home advantage effect.
The classes are imbalanced (draws are only ≈24%). Accuracy alone can mislead, so we should also report precision, recall and F1 per class.
Baseline: always predicting home_win gives ≈47% accuracy. Our model must beat this.

In [ ]:
# Distributions of numeric features (histograms)
num_cols = ["elo_home", "elo_away", "elo_diff", "stadium_capacity", "attendance_pct",
            "crowd_size", "travel_distance_km", "venue_familiarity", "referee_bias_susceptibility"]

df[num_cols].hist(bins=30, figsize=(14, 10), color="steelblue", edgecolor="white")
plt.suptitle("Distributions of numeric pre-match features", fontsize=14)
plt.tight_layout()
plt.show()

**Observations**
- `elo_home` and `elo_away` are spread between 1200 and 1900. `elo_diff` looks roughly normal and centred at 0.
- `attendance_pct` and `crowd_size` have a big spike at **0**: these are matches played **without a crowd** (`crowd_present = 0`, about 20% of matches).
- `travel_distance_km` is **right-skewed**: most trips are short, with a long tail up to ~2,400 km.
- Feature scales differ a lot (0–10 vs 0–90,000). A **Decision Tree does not need feature scaling** because it splits on one feature at a time using thresholds.

In [ ]:
# Categorical and binary features (count plots)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.countplot(data=df, y="league", order=df["league"].value_counts().index, ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Matches per league")

sns.countplot(data=df, x="crowd_present", ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Crowd present (0 = no, 1 = yes)")

sns.countplot(data=df, x="is_derby", ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Derby match (0 = no, 1 = yes)")

sns.countplot(data=df, x="timezones_crossed", ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Time zones crossed by away team")

plt.tight_layout()
plt.show()

**Observations**
- 8 leagues, all well represented (EPL has the most matches, Eredivisie the fewest).
- ~80% of matches had a crowd and only ~6% are derbies.
- Almost all away teams cross **0** time zones. Only ~3,900 matches cross 1, and just 2 matches cross 2 (too few to draw conclusions).
- `league` is text, so it will need **encoding** in preprocessing.

In [ ]:
# Relationships with the target (bar plots)
# The question here is: which features change how often the home team wins?
df["home_win"] = (df[target] == "home_win").astype(int)   # helper column for plotting only

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(data=df, x="crowd_present", y="home_win", ax=axes[0, 0], palette="Set2", errorbar=None)
axes[0, 0].set_title("Home win rate: crowd vs no crowd")

sns.barplot(data=df, x="is_derby", y="home_win", ax=axes[0, 1], palette="Set2", errorbar=None)
axes[0, 1].set_title("Home win rate: derby vs non-derby")

df["travel_group"] = pd.qcut(df["travel_distance_km"], 4,
                             labels=["Short", "Medium", "Long", "Very long"])
sns.barplot(data=df, x="travel_group", y="home_win", ax=axes[1, 0], palette="Set2", errorbar=None)
axes[1, 0].set_title("Home win rate by away team's travel distance")

league_order = df.groupby("league")["home_win"].mean().sort_values(ascending=False).index
sns.barplot(data=df, y="league", x="home_win", order=league_order, ax=axes[1, 1], palette="Set2", errorbar=None)
axes[1, 1].set_title("Home win rate by league")

for ax in axes.flat:
    ax.set_ylabel(ax.get_ylabel().replace("home_win", "Home win rate"))
    ax.set_xlabel(ax.get_xlabel().replace("home_win", "Home win rate"))
plt.tight_layout()
plt.show()

**Observations**
- **Crowd:** home teams win ≈48% with a crowd vs ≈43% without. Fans add a few percentage points.
- **Derbies:** home win rate drops to ≈43%. Local rivals travel little and know the venue, so the advantage shrinks.
- **Travel distance:** the farther the away team travels, the more often the home team wins (≈41% for the shortest trips vs ≈55% for the longest).
- **League:** win rates are almost identical (46–47%), so `league` alone is a weak predictor.

In [ ]:
# Team strength vs result
df["elo_group"] = pd.cut(df["elo_diff"], [-700, -300, -100, 100, 300, 700],
                         labels=["Much weaker", "Weaker", "Even", "Stronger", "Much stronger"])

result_by_elo = pd.crosstab(df["elo_group"], df[target], normalize="index")[order]
result_by_elo.plot(kind="bar", stacked=True, figsize=(10, 5), color=sns.color_palette("Set2"))
plt.title("Match result by home team's strength advantage (elo_diff)")
plt.xlabel("Home team compared to away team"); plt.ylabel("Share of matches")
plt.xticks(rotation=0); plt.legend(title="Result", bbox_to_anchor=(1, 1))
plt.show()

result_by_elo.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x=target, y="elo_diff", order=order, palette="Set2")
plt.title("elo_diff by match result")
plt.show()

**Observations**
- `elo_diff` is the **strongest single predictor**. When the home team is much stronger they win ≈74% of the time. When much weaker, only ≈21%.
- Even when the teams are **evenly matched**, the home team still wins ≈47% vs ≈29% for the away team. That gap is home advantage itself.
- The draw share stays around 17–25% in every group. Draws don't depend clearly on any feature, so they will be the **hardest class** for the model.

In [ ]:
# Correlation heatmap
corr_cols = [c for c in pre_match_features if c != "league"] + in_match_stats + ["home_win"]
corr = df[corr_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.7}, annot_kws={"size": 8})
plt.title("Correlation heatmap (pre-match features, in-match stats, home win)")
plt.tight_layout()
plt.show()

In [ ]:
corr["home_win"].drop("home_win").sort_values(ascending=False).round(3)

**Observations**
- Strongest correlations with a home win: `elo_diff` (+0.25), `elo_home` (+0.17), `elo_away` (−0.18), then `travel_distance_km` (+0.11).
- Crowd features (`attendance_pct`, `crowd_present`, `crowd_size`) and `venue_familiarity` have small positive correlations (~0.04).
- **Redundant features:** `elo_diff` = `elo_home − elo_away`, and `crowd_size` ≈ `stadium_capacity × attendance_pct`. They are highly correlated with each other. That is fine for a Decision Tree (it just picks the most useful one), but worth knowing.
- Correlation only measures **linear** relationships. A Decision Tree can also capture non-linear effects and interactions (e.g. travel × crowd).

In [ ]:
# Crowd and referee decisions
fouls_by_crowd = df.groupby("crowd_present")[["fouls_home", "fouls_away", "yellow_home", "yellow_away"]].mean()
fouls_by_crowd.index = ["No crowd", "Crowd"]

fouls_by_crowd.plot(kind="bar", figsize=(9, 5), color=sns.color_palette("Set2"))
plt.title("Average fouls and yellow cards: crowd vs no crowd")
plt.ylabel("Average per match"); plt.xticks(rotation=0)
plt.show()

fouls_by_crowd.round(2)

**Observations**
- **Without a crowd**, home and away teams are called for about the same number of fouls (≈11 each).
- **With a crowd**, away teams get more fouls (≈12.9 vs ≈9.4) and more yellow cards. This suggests crowds influence referees, which is one mechanism behind home advantage.
- We still won't use these columns as model features, because they are only known during the match.

# **(3) Data Preprocessing**.

In [ ]:
# Check missing values and duplicates
from sklearn.model_selection import train_test_split

data = df.copy()

print("Missing values:", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum())

# Drop leakage columns and the ID
data = data.drop(["match_id", "goals_home", "goals_away", "home_goal_diff", "home_points"], axis=1)

print("Shape after dropping:", data.shape)

# One-hot encode league
data = pd.get_dummies(data, columns=["league"])

league_cols = [c for c in data.columns if c.startswith("league_")]
data[league_cols] = data[league_cols].astype(int)

print(data.columns.tolist())
data.head()

# Split features and target
X = data.drop("home_result", axis=1)
y = data["home_result"]

print("Features:", X.columns.tolist())
print("Target:", y.name)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



# **(4) Model training**.

In [ ]:
# Features and target
features = [
    "league",
    "elo_home",
    "elo_away",
    "elo_diff",
    "crowd_present",
    "stadium_capacity",
    "attendance_pct",
    "crowd_size",
    "travel_distance_km",
    "timezones_crossed",
    "is_derby",
    "venue_familiarity",
    "referee_bias_susceptibility"
]

X = df[features]
y = df["home_result"]

# Remove rows with missing target
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]

# Identify categorical and numerical features
categorical_features = ["league"]
numerical_features = [col for col in features if col != "league"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Encode categorical features
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", "passthrough", numerical_features)
])

# Build, train the model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

# **(5) Model Evaluation**.

## Decision Tree — Impurity Measures

A Decision Tree chooses each split by selecting the feature and threshold that produce the **purest** child nodes. Purity is measured using the equations below.

Let a node contain samples from $C$ classes, and let $p_i$ be the proportion of samples belonging to class $i$ in that node:

$$p_i = \frac{\text{number of samples of class } i}{\text{total samples in the node}}$$

### 1. Gini Impurity

$$G = 1 - \sum_{i=1}^{C} p_i^{\,2}$$

- $G = 0$ means the node is perfectly pure (all samples belong to one class).
- $G$ is maximised when the classes are evenly mixed.
- This is the default criterion in scikit-learn (`criterion="gini"`).

### 2. Entropy

$$H = - \sum_{i=1}^{C} p_i \, \log_2(p_i)$$

- $H = 0$ for a perfectly pure node.
- Entropy measures the amount of disorder (uncertainty) in the node.
- Used when `criterion="entropy"`.

### 3. Information Gain

When a node (the *parent*) is split into $k$ child nodes, the quality of the split is measured by the reduction in impurity. Using entropy:

$$IG = H_{\text{parent}} - \sum_{j=1}^{k} \frac{N_j}{N} \, H_j$$

where:
- $N$ = number of samples in the parent node,
- $N_j$ = number of samples in child node $j$,
- $H_j$ = entropy of child node $j$,
- $\dfrac{N_j}{N}$ = the weight of each child (its share of the samples).

The tree selects the split that **maximises** $IG$ (equivalently, minimises the weighted child impurity). The same weighted formula applies to Gini:

$$G_{\text{split}} = \sum_{j=1}^{k} \frac{N_j}{N} \, G_j$$

### 4. Prediction at a Leaf

Once splitting stops, each leaf predicts the **majority class** among its samples:

$$\hat{y} = \arg\max_{i} \; p_i$$



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support,
                             roc_curve, roc_auc_score)
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

labels = ["home_win", "draw", "away_win"]
colors = {"home_win": "#6B9E78", "draw": "#C9A24D", "away_win": "#C07A7A"}

# BaseLine
We compare the model with a simple baseline that always predicts the most common result (home win). The model is only useful if it beats this.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_base = baseline.predict(X_test)

In [ ]:
print("Baseline accuracy:", round(accuracy_score(y_test, y_base), 3))
print("Baseline macro F1:", round(f1_score(y_test, y_base, average="macro"), 3))
print()
print("Model accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Model macro F1:", round(f1_score(y_test, y_pred, average="macro"), 3))